# Overlap between the activity and the pose dataset (R1.7, R2.2, R1.3)

- **R1.7** asks for the overlap in ligands, kinase targets and exact kinase–ligand pairs.
- **R2.2** asks whether a complex in one task's test set can appear in the other task's training data.
- **R1.3** needs complexes that have **both** an activity value and a **true** RMSD. Only the pose (cross-docking) data have a true RMSD (docked pose vs. the ligand's crystal pose). The activity (Kinodata-3D) data have only a predicted RMSD. Such complexes would exist only where the two datasets share a kinase–ligand pair.

**Identity used:**
- **Ligand:** InChIKey computed with RDKit from each dataset's SMILES (activity: ChEMBL; pose: KLIFS). Matches are reported for the full key and for its first block (connectivity, which ignores stereo, charge and isotopes).
- **Kinase:** the 85-residue KLIFS pocket sequence. Pocket sequences are per structure and contain gaps (`_`) for missing residues. Two pockets count as the same kinase when they agree at every position where both have a residue (and share at least 70 such positions). Pockets are reported as counted, not mapped to UniProt.

Input: the per-complex tables cached by `rmsd_distribution.ipynb` (run that notebook first).

In [1]:
from pathlib import Path

# repo root = first parent folder that contains the kinodata package
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kinodata" / "__init__.py").exists())
OUTPUT = REPO / "analysis" / "output"
OUTPUT.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

RMSD_MAX = 10   # reference configuration; None = all complexes with an RMSD
MIN_POCKET_OVERLAP = 70

act_file, pose_file = OUTPUT / "rmsd_activity.csv.gz", OUTPUT / "rmsd_pose.csv.gz"
if not (act_file.exists() and pose_file.exists()):
    raise FileNotFoundError("run analysis/rmsd_distribution.ipynb first: it caches the per-complex tables")
activity = pd.read_csv(act_file)
pose = pd.read_csv(pose_file)
pose = pose[pose["predicted_rmsd"].notna()]
if RMSD_MAX is not None:
    activity = activity[activity["predicted_rmsd"] <= RMSD_MAX].copy()
    pose = pose[pose["predicted_rmsd"] <= RMSD_MAX].copy()
print(f"activity: {len(activity):,} complexes, pose: {len(pose):,} poses (RMSD <= {RMSD_MAX} A)")

activity: 119,421 complexes, pose: 67,976 poses (RMSD <= 10 A)


In [2]:
def inchikey(smiles):
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    return Chem.MolToInchiKey(mol) if mol is not None else None


keys = {s: inchikey(s) for s in pd.unique(pd.concat([activity["smiles"], pose["smiles"]]))}
for name, df in [("activity", activity), ("pose", pose)]:
    df["inchikey"] = df["smiles"].map(keys)
    df["connectivity"] = df["inchikey"].str[:14]
    print(f"{name}: {df['inchikey'].isna().sum():,} SMILES RDKit could not parse")

activity: 0 SMILES RDKit could not parse
pose: 0 SMILES RDKit could not parse


In [3]:
def same_pocket(a, b, min_overlap=MIN_POCKET_OVERLAP):
    """Same kinase: identical residues wherever both KLIFS pocket sequences have one."""
    if not (isinstance(a, str) and isinstance(b, str)) or len(a) != len(b):
        return False
    both = [(x, y) for x, y in zip(a, b) if x not in "_-" and y not in "_-"]
    return len(both) >= min_overlap and all(x == y for x, y in both)


act_pockets = activity["pocket_sequence"].dropna().unique()
pose_pockets = pose["pocket_sequence"].dropna().unique()
pocket_match = {(a, b): same_pocket(a, b) for a in act_pockets for b in pose_pockets}
act_with_pose = {a for (a, b), m in pocket_match.items() if m}
pose_with_act = {b for (a, b), m in pocket_match.items() if m}

rows = []
for level in ["inchikey", "connectivity"]:
    la, lp = set(activity[level].dropna()), set(pose[level].dropna())
    rows.append({"level": f"ligands ({level})", "activity": len(la), "pose": len(lp), "shared": len(la & lp)})
rows.append({"level": "kinase pockets (identical sequence)", "activity": len(act_pockets), "pose": len(pose_pockets),
             "shared": len(set(act_pockets) & set(pose_pockets))})
rows.append({"level": "kinase pockets (same kinase, gaps ignored)", "activity": len(act_pockets), "pose": len(pose_pockets),
             "shared": f"{len(act_with_pose)} activity / {len(pose_with_act)} pose pockets"})
pd.DataFrame(rows)

,level,activity,pose,shared
0,ligands (inchikey),87158,1531,739
1,ligands (connectivity),83635,1469,748
2,kinase pockets (identical sequence),850,131,111
3,"kinase pockets (same kinase, gaps ignored)",850,131,369 activity / 114 pose pockets


## Shared kinase–ligand pairs

A pair is shared when the same ligand (connectivity block of the InChIKey) is bound to the same kinase (compatible pocket sequence) in both datasets. For each shared pair the table gives the activity complexes (pIC50 values, predicted RMSD of the Kinodata-3D pose) and the cross-docked poses (number, best and median **true** RMSD).

In [4]:
shared_ligands = set(activity["connectivity"].dropna()) & set(pose["connectivity"].dropna())
a = activity[activity["connectivity"].isin(shared_ligands)]
p = pose[pose["connectivity"].isin(shared_ligands)]
cand = a.reset_index().merge(p.reset_index(), on="connectivity", suffixes=("_act", "_pose"))
cand = cand[[pocket_match.get((x, y), False) for x, y in zip(cand["pocket_sequence_act"], cand["pocket_sequence_pose"])]]

pairs = (cand.groupby(["connectivity", "pocket_sequence_act"])
         .agg(activity_complexes=("index_act", "nunique"), pIC50=("y", lambda s: ", ".join(f"{v:.2f}" for v in sorted(set(s.round(2))))),
              kinodata_pred_rmsd=("predicted_rmsd_act", "min"), poses=("index_pose", "nunique"),
              best_true_rmsd=("predicted_rmsd_pose", "min"), median_true_rmsd=("predicted_rmsd_pose", "median"))
         .reset_index())
n_act = pairs["activity_complexes"].sum()
print(f"shared ligands (connectivity): {len(shared_ligands):,}")
print(f"shared kinase-ligand pairs: {len(pairs):,}, covering {n_act:,} activity complexes "
      f"({100 * n_act / len(activity):.2f} % of the activity data) and {pairs['poses'].sum():,} poses")
pairs.drop(columns=["pocket_sequence_act"]).sort_values("activity_complexes", ascending=False).head(30)

shared ligands (connectivity): 748
shared kinase-ligand pairs: 562, covering 611 activity complexes (0.51 % of the activity data) and 22,069 poses


,connectivity,activity_complexes,pIC50,kinodata_pred_rmsd,poses,best_true_rmsd,median_true_rmsd
330,PRWQQLFMBINSAW,4,"6.74, 6.80, 7.89, 8.52",1.488895,18,1.112862,4.959192
126,HDYFVAJZXGMYBL,4,"6.60, 7.17, 7.85, 9.00",3.824681,134,0.683154,2.586977
392,SEGFWFQHVPYNMS,4,"7.49, 8.14, 8.49, 8.62",0.756079,60,0.585250,3.435910
428,UPBRYBJZTWHUQK,3,"7.33, 7.37, 9.00",1.413337,18,0.951919,3.963722
47,CDISOBGWDHAHGF,3,"7.81, 8.26, 10.08",2.740265,1,0.509310,0.509310
525,YLJNGHKLWZPDSX,2,"6.08, 8.34",0.893587,43,0.269563,5.407268
393,SHENXRCDBJGWNU,2,"8.03, 8.62",0.883435,43,0.687076,2.055106
74,DNXZIJKGPFJRAA,2,"7.55, 8.30",1.383101,131,0.753158,2.691732
249,MLNKSPKBBXWPSG,2,8.70,1.823690,85,0.587424,2.615468
438,UXZRCIBKIDILEF,2,"6.77, 8.15",2.123687,61,1.138585,4.768117


In [5]:
if len(pairs):
    print("best true RMSD of the shared pairs (cross-docked poses):")
    display(pairs["best_true_rmsd"].describe().round(2).to_frame().T)
    print(f"pairs with at least one cross-docked pose <= 2.5 A: {(pairs['best_true_rmsd'] <= 2.5).sum():,}")

best true RMSD of the shared pairs (cross-docked poses):


,count,mean,std,min,25%,50%,75%,max
best_true_rmsd,562.0,1.1,1.2,0.08,0.45,0.75,1.2,9.99


pairs with at least one cross-docked pose <= 2.5 A: 522


## Save the shared pairs

One row per activity complex that belongs to a shared kinase–ligand pair. `ident` and `smiles` identify the Kinodata-3D complex, so it can be found in the split files (`activity_smiles_split_<k>.csv`) and test predictions of a run, for example to see how a model did on these structurally referenced complexes (R1.6, R3.10). The pose-side columns summarize the cross-docked poses of the same pair.

In [6]:
pose_side = pairs.set_index(["connectivity", "pocket_sequence_act"])[["poses", "best_true_rmsd", "median_true_rmsd"]]
shared_complexes = (cand.drop_duplicates("index_act")
                    .rename(columns={"y": "pIC50", "predicted_rmsd_act": "kinodata_pred_rmsd", "smiles_act": "smiles",
                                     "inchikey_act": "inchikey", "ident_act": "ident", "pocket_sequence_act": "pocket_sequence"})
                    [["ident", "smiles", "inchikey", "connectivity", "pocket_sequence", "pIC50", "kinodata_pred_rmsd"]]
                    .join(pose_side, on=["connectivity", "pocket_sequence"]))
shared_complexes.to_csv(OUTPUT / "shared_pairs.csv", index=False)
print(f"{len(shared_complexes):,} activity complexes written to {OUTPUT / 'shared_pairs.csv'}")
shared_complexes.head()

611 activity complexes written to /data1/choderaj/lopezrr/kinodata-3D-affinity-prediction/analysis/output/shared_pairs.csv


,ident,smiles,inchikey,connectivity,pocket_sequence,pIC50,kinodata_pred_rmsd,poses,best_true_rmsd,median_true_rmsd
170,6656,O=C(Nc1ccc(Oc2ccccc2)cc1)c1cccnc1NCc1ccncc1,CPVRYQAOUPSUDO-UHFFFAOYSA-N,CPVRYQAOUPSUDO,KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVE...,7.420216,0.694036,11,1.440058,2.102255
221,95063,Nc1ncnc2c1c(-c1ccc(Oc3ccccc3)cc1)cn2C1CCOCC1,BCWVVERKADMJGG-UHFFFAOYSA-N,BCWVVERKADMJGG,KKLGAGQFGEVWMVAVKTMAFLAEANVMKTLQDKLVKLHAVYIITE...,7.508638,1.120511,20,0.368781,1.119888
361,10580,CCC(=O)Nc1ccc2ncnc(Nc3cccc(Br)c3)c2c1,WUPUZEMRHDROEO-UHFFFAOYSA-N,WUPUZEMRHDROEO,KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...,9.683266,2.752977,29,1.197264,5.439753
423,42476,Fc1cccc([C@H]2CCCN2c2ccc3[nH+]cc(-c4ccccn4)n3n...,RKERQEKFTWWCEY-GOSISDBHSA-O,RKERQEKFTWWCEY,WELGEGAFGKVFLVAVKALDFQREAELLTMLQQHIVRFFGVLMVFE...,8.522879,0.943645,3,2.258478,4.904541
666,99108,COc1cccc(OCc2cc(C)ccn2)c1-c1ccc2c(c1)CC[NH2+]CC2,KYANYGKXMNYFBX-UHFFFAOYSA-O,KYANYGKXMNYFBX,KELGSGNFGTVKKVAVKILELLAEANVMQQLDPYIVRMIGIMLVME...,7.100015,1.675317,48,0.504787,2.432127
